# 05 Chart And Signal Scanner

Notebook n?y d?nh cho ph?n tr?c quan h?a chi?n l??c Combo:
- scan t?n hi?u reversal cho m?t ho?c nhi?u symbol
- xem b?ng signal
- v? chart Plotly cho symbol ?ang ch?n


In [ ]:
# Bootstrap: add repo root + core_python to sys.path
import sys
from pathlib import Path

def _find_root(start: Path, marker: str = 'config.py') -> Path:
    for p in [start, *start.parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f'Could not locate repo root containing {marker!r}')

ROOT = _find_root(Path.cwd())
CORE = ROOT / 'core_python'
for p in (str(ROOT), str(CORE)):
    if p not in sys.path:
        sys.path.insert(0, p)

print('ROOT =', ROOT)
print('CORE =', CORE)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('dark_background')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

from strategies.combo.config import SYMBOLS, get_indicator_params, summary as strategy_summary
from strategies.combo.scanner import (
    build_reversal_figure,
    calc_reversal_stats,
    run_multi_reversal_scan,
    run_reversal_scan,
)

print(strategy_summary())
print('Symbols:', ', '.join(SYMBOLS.keys()))


In [ ]:
SCAN_SYMBOL = 'US30'
SCAN_SYMBOLS = ['US30', 'US500', 'DE40', 'GOLD', 'BTCUSD']
N_BARS = 200

SCAN_PARAMS = get_indicator_params()
SCAN_PARAMS.update({
    # 'MA_PERIOD': 20,
    # 'KTP': 2.3,
    # 'MIN_RR': 1.25,
})
SCAN_PARAMS


In [ ]:
single = run_reversal_scan(SCAN_SYMBOL, N_BARS, SCAN_PARAMS)
print('Single-symbol stats:', calc_reversal_stats(single['signals_df']))
display(single['signals_df'].tail(20))


In [ ]:
fig = build_reversal_figure(SCAN_SYMBOL, single, SCAN_PARAMS)
fig.show()


In [ ]:
multi = run_multi_reversal_scan(SCAN_SYMBOLS, N_BARS, SCAN_PARAMS)
summary_rows = []
for sym, result in multi.items():
    row = {'symbol': sym, **calc_reversal_stats(result['signals_df'])}
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows).sort_values(['n_pass', 'win_pct', 'avg_rr'], ascending=False, ignore_index=True)
display(summary_df)


In [ ]:
selected = summary_df.iloc[0]['symbol'] if not summary_df.empty else SCAN_SYMBOL
print('Best current scanner candidate =', selected)
selected_result = multi.get(selected) or single
selected_fig = build_reversal_figure(selected, selected_result, SCAN_PARAMS)
selected_fig.show()
